# 02 - Tools Customizadas
> Conectando Claude a banco de dados e APIs internas

**Modulo:** `03_tool_use` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE produtos (id INT, nome TEXT, preco REAL, estoque INT)')
conn.executemany('INSERT INTO produtos VALUES (?,?,?,?)',
    [(1,'Notebook',3500,10),(2,'Mouse',150,45),(3,'Monitor',1800,8)])
conn.commit()

TOOLS = [{'name':'buscar_produto',
    'description':'Busca produtos por nome ou ID.',
    'input_schema':{'type':'object','properties':{'busca':{'type':'string'}},'required':['busca']}}]

def buscar_produto(busca):
    try: rows=conn.execute('SELECT * FROM produtos WHERE id=?',(int(busca),)).fetchall()
    except: rows=conn.execute('SELECT * FROM produtos WHERE nome LIKE ?',(f'%{busca}%',)).fetchall()
    return json.dumps([{'id':r[0],'nome':r[1],'preco':r[2],'estoque':r[3]} for r in rows])

def loop(q):
    msgs=[{'role':'user','content':q}]
    while True:
        r=client.messages.create(model=HAIKU,max_tokens=512,tools=TOOLS,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                res.append({'type':'tool_result','tool_use_id':b.id,'content':buscar_produto(**b.input)})
        msgs.append({'role':'user','content':res})

print(loop('Quais produtos tem menos de 15 unidades em estoque?'))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
